# Part 1 — Full re-baseline: CatBoost + Dirichlet, all models, all seeds, all 36 datasets
**Calibration vs Deep Ensembles, MAKE revision**

## What this notebook does

1. Loads the exact 36 datasets from `dataset_manifest.csv`
2. Trains each model **once** per (dataset, seed) and applies **5 calibrators** to the saved predictions (4× speedup vs your original code)
3. **Re-uses any probs already cached** in `WORK_DIR/probs/` — already-computed cells don't retrain
4. Saves all probabilities so notebooks 02 & 03 are nearly free
5. Checkpoints to `WORK_DIR` after every dataset

**Expected wall-clock:** 4–6 h actual. Budget 8–10 h of wall-clock time.

**Output:** `results_raw.csv` with **4,500 rows** (5 models × 5 calibrators × 5 seeds × 36 datasets), all from Colab hardware.

## 1. Environment setup

In [ ]:
# Working directory for cached probs and intermediate CSVs.
# Override with:  export WORK_DIR=/path/to/persistent/storage
import os, pathlib

WORK_DIR = pathlib.Path(os.environ.get('WORK_DIR', './work')).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROBS_DIR = WORK_DIR / 'probs'
PROBS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Cached probability files: {len(list(PROBS_DIR.glob('*.npz')))}")


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    print("  TF32 + cudnn.benchmark enabled.")
else:
    print("WARNING: no GPU detected.")

In [ ]:
# Install dependencies. CatBoost and dirichletcal are new; rest match your requirements.txt
!pip install catboost dirichletcal
!pip install openml==0.15.1 lightgbm==4.3.0 'xgboost>=2.0.0' 'scikit-learn>=1.3.0'

# Verify
import catboost, dirichletcal
print(f"\n✓ catboost {catboost.__version__}")
print(f"✓ dirichletcal {dirichletcal.__version__}")

## 2. Get your codebase + manifest

In [ ]:
# Locate the repo root (the directory containing src/) and add it to sys.path.
# Works regardless of whether the notebook is run from notebooks/, the repo root,
# or one level deeper.
import os, pathlib, sys

def _find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src').is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find a 'src/' directory in the current working directory or any "
        "parent. Run this notebook from inside the cloned repository."
    )

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")
print("✓ src/ found")


In [ ]:
# Make src/ importable and verify
import sys
sys.path.insert(0, os.getcwd())

from src.datasets import load_task, split_dataset
from src.calibration import build_calibrator, CALIBRATOR_LABELS, BaseCalibrator
from src.metrics import evaluate_all
from src.models import LightGBMModel, XGBoostModel, SingleMLP, DeepEnsemble, get_device
from src.utils import EPS
print("✓ src/ imports OK")

In [ ]:
# Load dataset_manifest.csv. Looks in WORK_DIR first, then data/ in the repo.
import pathlib, shutil
import pandas as pd

manifest_candidates = [
    WORK_DIR / 'dataset_manifest.csv',
    REPO_ROOT / 'data' / 'dataset_manifest.csv',
]
manifest_path = next((p for p in manifest_candidates if p.exists()), None)

if manifest_path is None:
    raise FileNotFoundError(
        "dataset_manifest.csv not found. Place it in $WORK_DIR or in the repo's "
        f"data/ directory. Searched: {[str(p) for p in manifest_candidates]}"
    )

# Copy into WORK_DIR if it was found elsewhere, so later cells use a single location.
if manifest_path != WORK_DIR / 'dataset_manifest.csv':
    shutil.copy(manifest_path, WORK_DIR / 'dataset_manifest.csv')

manifest = pd.read_csv(WORK_DIR / 'dataset_manifest.csv')
print(f"✓ Manifest: {len(manifest)} datasets")
assert len(manifest) == 36, f"Expected 36 datasets, got {len(manifest)}"
print(manifest[['name', 'n_samples', 'n_classes', 'size_regime']].to_string(index=False))


## 4. New model: CatBoost


In [ ]:
import numpy as np
from catboost import CatBoostClassifier
from src.utils import EPS

class CatBoostModel:
    """CatBoost classifier. AutoML defaults, matched to LightGBM/XGBoost spirit."""

    _DEFAULT_PARAMS = {
        "learning_rate":        0.05,
        "depth":                6,
        "min_data_in_leaf":     20,
        "rsm":                  0.8,
        "subsample":            0.8,
        "bootstrap_type":       "Bernoulli",
        "l2_leaf_reg":          3.0,
        "od_type":              "Iter",
        "od_wait":              50,
        "verbose":              False,
        "thread_count":         -1,
        "allow_writing_files":  False,
    }

    def __init__(self, n_classes, num_boost_round=500, extra_params=None):
        self.n_classes = n_classes
        params = dict(self._DEFAULT_PARAMS)
        params["iterations"] = num_boost_round
        if n_classes > 2:
            params["loss_function"]  = "MultiClass"
            params["eval_metric"]    = "MultiClass"
            params["classes_count"]  = n_classes
        else:
            params["loss_function"]  = "Logloss"
            params["eval_metric"]    = "Logloss"
        if extra_params:
            params.update(extra_params)
        self._params = params
        self._model  = None

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self._model = CatBoostClassifier(**self._params)
        eval_set = (X_val, y_val) if X_val is not None else None
        self._model.fit(X_train, y_train, eval_set=eval_set, verbose=False, plot=False)
        return self

    def predict_proba(self, X):
        assert self._model is not None
        return np.clip(self._model.predict_proba(X), EPS, 1.0)

# Smoke test
from sklearn.datasets import make_classification
Xtr, ytr = make_classification(n_samples=200, n_features=10, n_classes=3, n_informative=5, random_state=0)
m = CatBoostModel(n_classes=3, num_boost_round=50)
m.fit(Xtr, ytr)
p = m.predict_proba(Xtr[:5])
print(f"✓ CatBoost smoke test passed. Shape: {p.shape}, sum-to-1: {p.sum(axis=1).round(6)}")

## 5. New calibrator: Dirichlet (ODIR)


In [ ]:
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

class DirichletODIRCalibrator(BaseCalibrator):
    """Dirichlet calibration (ODIR variant) — Kull et al. 2019."""

    def __init__(self, reg_lambda=1e-3, reg_mu=1e-3):
        self.reg_lambda = reg_lambda
        self.reg_mu     = reg_mu
        self._model     = None

    def fit(self, probs_val, y_val):
        self._model = FullDirichletCalibrator(
            reg_lambda=self.reg_lambda,
            reg_mu=self.reg_mu,
        )
        self._model.fit(probs_val, y_val)
        return self

    def calibrate(self, probs):
        return np.clip(self._model.predict_proba(probs), EPS, 1.0 - EPS)

    def __repr__(self):
        return f"DirichletODIRCalibrator(reg_lambda={self.reg_lambda}, reg_mu={self.reg_mu})"

# Smoke test
from sklearn.linear_model import LogisticRegression
Xtr, ytr = make_classification(n_samples=500, n_features=10, n_classes=3, n_informative=5, random_state=0)
clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
p_val = clf.predict_proba(Xtr[:200])
p_test = clf.predict_proba(Xtr[200:])
cal = DirichletODIRCalibrator().fit(p_val, ytr[:200])
p_cal = cal.calibrate(p_test)
print(f"✓ Dirichlet smoke test passed. Shape: {p_cal.shape}")

## 6. Refactored runner — train once, calibrate many

Each (dataset, seed, model) trains exactly once. Probabilities are cached to
`WORK_DIR/probs/` as `.npz` files. Subsequent calibrators sweep over the cached
predictions for free.

**This means:** any probs already in `WORK_DIR/probs/` from a prior run will be
reused — those cells don't retrain.

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

def build_model(model_name, n_classes, n_features, seed, device):
    if model_name == "lgbm":
        return LightGBMModel(n_classes=n_classes), {}
    if model_name == "xgboost":
        return XGBoostModel(n_classes=n_classes), {}
    if model_name == "catboost":
        return CatBoostModel(n_classes=n_classes), {}
    if model_name == "single_mlp":
        return SingleMLP(
            input_dim=n_features, n_classes=n_classes,
            hidden_dims=(256, 128), lr=1e-3, epochs=200,
            batch_size=256, patience=20, device=device,
        ), {"seed": seed}
    if model_name == "deep_ensemble":
        return DeepEnsemble(
            n_members=5,
            input_dim=n_features, n_classes=n_classes,
            hidden_dims=(256, 128), lr=1e-3, epochs=200,
            batch_size=256, patience=20, device=device,
        ), {"base_seed": seed}
    raise ValueError(f"Unknown model: {model_name}")


def build_calibrator_extended(name):
    if name == "dirichlet":
        return DirichletODIRCalibrator()
    return build_calibrator(name)


def probs_path(task_id, model_name, seed):
    return PROBS_DIR / f"{task_id}__{model_name}__seed{seed}.npz"


def get_predictions(split, model_name, seed, device):
    """Returns cached or freshly-trained predictions for one (task, model, seed) cell."""
    task_id = split["task_id"]
    path = probs_path(task_id, model_name, seed)

    if path.exists():
        data = np.load(path)
        return (data["probs_val_cal"], data["probs_test"],
                float(data["train_time_s"]), data["y_val_cal"], data["y_test"])

    model, fit_kwargs = build_model(
        model_name, split["n_classes"], split["n_features"], seed, device
    )

    t0 = time.time()
    model.fit(split["X_train"], split["y_train"],
              split["X_val_es"], split["y_val_es"], **fit_kwargs)
    train_time = time.time() - t0

    probs_val_cal = model.predict_proba(split["X_val_cal"])
    probs_test    = model.predict_proba(split["X_test"])

    np.savez_compressed(
        path,
        probs_val_cal=probs_val_cal, probs_test=probs_test,
        y_val_cal=split["y_val_cal"], y_test=split["y_test"],
        train_time_s=np.array(train_time),
    )
    return probs_val_cal, probs_test, train_time, split["y_val_cal"], split["y_test"]


def evaluate_cell(split, model_name, calibrator_name, seed, device,
                  train_time, probs_val_cal, probs_test, y_val_cal, y_test):
    calibrator = build_calibrator_extended(calibrator_name)
    calibrator.fit(probs_val_cal, y_val_cal)
    probs_cal = calibrator.calibrate(probs_test)

    metrics     = evaluate_all(probs_cal,  y_test)
    raw_metrics = evaluate_all(probs_test, y_test)

    cal_label = (CALIBRATOR_LABELS.get(calibrator_name)
                 if calibrator_name in CALIBRATOR_LABELS
                 else "Dirichlet ODIR")

    return {
        "task_id":          split["task_id"],
        "dataset_name":     split["name"],
        "n_samples":        split["n_samples"],
        "n_train":          split["n_train"],
        "n_val_es":         split["n_val_es"],
        "n_val_cal":        split["n_val_cal"],
        "n_test":           split["n_test"],
        "n_features":       split["n_features"],
        "n_classes":        split["n_classes"],
        "seed":             seed,
        "model":            model_name,
        "calibrator":       calibrator_name,
        "calibrator_label": cal_label,
        **{f"cal_{k}": v for k, v in metrics.items()},
        **{f"raw_{k}": v for k, v in raw_metrics.items()},
        "train_time_s":     round(train_time, 2),
    }

print("✓ Refactored runner defined.")

## 7. Configuration

In [ ]:
# Datasets locked from manifest — no re-roll
TASK_IDS = manifest["task_id"].tolist()
assert len(TASK_IDS) == 36
print(f"✓ {len(TASK_IDS)} task IDs locked from manifest")

# 5 models, 5 calibrators, 5 seeds
MODELS      = ["lgbm", "xgboost", "catboost", "single_mlp", "deep_ensemble"]
CALIBRATORS = ["none", "temp", "logistic", "isotonic", "dirichlet"]
SEEDS       = [0, 1, 2, 3, 4]

device = get_device()
print(f"Device: {device}")

expected_rows = len(MODELS) * len(CALIBRATORS) * len(SEEDS) * len(TASK_IDS)
expected_trainings = len(MODELS) * len(SEEDS) * len(TASK_IDS)
print(f"Target total rows: {expected_rows} ({len(MODELS)} × {len(CALIBRATORS)} × {len(SEEDS)} × {len(TASK_IDS)})")
print(f"Target model trainings: {expected_trainings}")

# Already-cached probs from previous (validation) session
cached = set()
for p in PROBS_DIR.glob('*.npz'):
    # filename: <task_id>__<model>__seed<seed>.npz
    stem = p.stem
    parts = stem.split('__')
    if len(parts) == 3:
        tid, model_name, seed_str = parts
        cached.add((int(tid), model_name, int(seed_str.replace('seed', ''))))
print(f"Probabilities already cached from previous sessions: {len(cached)} (task, model, seed) combos")
print(f"  → these {len(cached)} model trainings will be skipped (cached probs reused)")

## 8. Run benchmark — full re-baseline

Outer loop is (dataset, seed) so a Colab disconnect loses at most one (dataset, seed) cell.
The partial CSV is written to `WORK_DIR` after every dataset and reloaded on resume.

In [ ]:
from datetime import datetime
import logging

logging.basicConfig(level=logging.WARNING)

# Resume from partial checkpoint if present
partial_ckpt = WORK_DIR / "part1_partial.csv"
if partial_ckpt.exists():
    prev = pd.read_csv(partial_ckpt)
    new_results = prev.to_dict("records")
    done_keys = {(int(r["task_id"]), r["model"], r["calibrator"], int(r["seed"]))
                 for r in new_results}
    print(f"Resumed: {len(new_results)} rows already done.")
else:
    new_results = []
    done_keys = set()

fail_log = []
total_cells = len(TASK_IDS) * len(SEEDS) * len(MODELS) * len(CALIBRATORS)
processed = len(done_keys)
start_time = time.time()

for task_id in TASK_IDS:
    print(f"\n── Task {task_id} ────────────────────────────────")
    data = load_task(task_id)
    if data is None:
        print(f"  ⚠ skipped (load failed)")
        fail_log.append({"task_id": task_id, "reason": "load_task returned None"})
        continue
    print(f"  {data['name']}  n={data['n_samples']}  d={data['n_features']}  K={data['n_classes']}")

    for seed in SEEDS:
        split = split_dataset(data, seed=seed)

        for model_name in MODELS:
            try:
                probs_vc, probs_te, train_time, y_vc, y_te = get_predictions(
                    split, model_name, seed, device
                )
            except Exception as exc:
                print(f"    ✗ {model_name} seed={seed} TRAIN FAILED: {exc}")
                fail_log.append({"task_id": task_id, "model": model_name, "seed": seed,
                                 "reason": f"train: {exc}"})
                continue

            for cal_name in CALIBRATORS:
                key = (task_id, model_name, cal_name, seed)
                if key in done_keys:
                    continue
                try:
                    row = evaluate_cell(split, model_name, cal_name, seed, device,
                                        train_time, probs_vc, probs_te, y_vc, y_te)
                    new_results.append(row)
                    done_keys.add(key)
                    processed += 1
                except Exception as exc:
                    print(f"    ✗ {model_name}/{cal_name} seed={seed} CAL FAILED: {exc}")
                    fail_log.append({"task_id": task_id, "model": model_name,
                                     "seed": seed, "calibrator": cal_name,
                                     "reason": f"calibrate: {exc}"})

    # Checkpoint to WORK_DIR after every dataset
    if new_results:
        pd.DataFrame(new_results).to_csv(partial_ckpt, index=False)

    elapsed = time.time() - start_time
    pct = 100 * processed / total_cells
    eta_s = elapsed / max(processed, 1) * (total_cells - processed)
    print(f"  ✓ task done. {processed}/{total_cells} cells ({pct:.1f}%) | "
          f"elapsed {elapsed/60:.1f}min | ETA {eta_s/60:.1f}min")

total_min = (time.time() - start_time) / 60
print(f"\n══════════════════════════════════════")
print(f"DONE. Rows produced: {len(new_results)}")
print(f"Failures: {len(fail_log)}")
print(f"Total wall-clock: {total_min:.1f} min ({total_min/60:.2f} h)")
if fail_log:
    pd.DataFrame(fail_log).to_csv(WORK_DIR / "part1_failures.csv", index=False)
    print(f"Failures logged to {WORK_DIR}/part1_failures.csv")

## 9. Save final results and summarize

In [ ]:
final_df = pd.DataFrame(new_results)
print(f"Rows: {len(final_df)} (target: {expected_rows})")

# Sanity matrix
matrix = final_df.groupby(["model", "calibrator"]).size().unstack(fill_value=0)
print("\nRow count by (model, calibrator) — each cell should be 5 seeds × 36 datasets = 180:")
print(matrix)

# Save final
final_path = WORK_DIR / "results_raw.csv"
final_df.to_csv(final_path, index=False)
print(f"\n✓ Final results: {final_path}")

# Remove partial since final is saved
if partial_ckpt.exists():
    partial_ckpt.unlink()
    print("✓ Partial checkpoint cleaned up.")

In [ ]:
# Summary of new baseline
summary = (final_df
    .groupby(["model", "calibrator"])
    [["cal_nll", "cal_ece_mean", "cal_brier_score", "cal_accuracy"]]
    .median()
    .round(4))

print("Per-(model, calibrator) median across 36 datasets × 5 seeds:")
print(summary.to_string())

print("\n── CatBoost vs LightGBM (Temp Scaling) ──")
for m in ["cal_nll", "cal_ece_mean", "cal_brier_score"]:
    lgb = summary.loc[("lgbm", "temp"), m]
    cat = summary.loc[("catboost", "temp"), m]
    print(f"  {m:18s}  LGB={lgb:.4f}  Cat={cat:.4f}  diff={cat-lgb:+.4f}")

print("\n── Dirichlet ODIR across model families ──")
for model in MODELS:
    raw = summary.loc[(model, "none"), "cal_nll"]
    dir_ = summary.loc[(model, "dirichlet"), "cal_nll"]
    mlr  = summary.loc[(model, "logistic"), "cal_nll"]
    print(f"  {model:18s}  raw={raw:.4f}  Dirichlet={dir_:.4f}  MLR={mlr:.4f}")